# 2. Train the surrogate and compare it with the baselines

Reads the committed results. Nothing here recomputes a number that appears in
the documentation — the tables *are* the evidence, and this notebook reads
them so a figure can never disagree with a table.

Regenerate with `python scripts/compare_methods.py --config configs/base.yaml`.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "2")
import sys
sys.path.insert(0, "../src")
import numpy as np, pandas as pd, torch
torch.set_num_threads(2)
import matplotlib.pyplot as plt
pd.set_option("display.width", 130)


In [ ]:
T = '../results/tables/'
methods = pd.read_csv(T+'method_comparison.csv')
print(f"{methods.method.nunique()} methods x {methods.split.nunique()} splits")
cols = ['split','method','mae','rmse','spearman_pooled','spearman_within_scenario','top1_agreement']
methods.loc[methods.split=='test_id', cols].round(4).to_string(index=False)

## Magnitude error and rank fidelity are different questions

The target is 94% exact zeros, so a model that predicts near-zero everywhere
gets a good MAE and is useless for screening. Rank fidelity is the metric that
matters for the actual task, and the two columns below can disagree sharply.

In [ ]:
sub = methods[methods.split=='test_id'].set_index('method')
fig, axes = plt.subplots(1,2, figsize=(12,4))
sub['mae'].sort_values().plot.barh(ax=axes[0], color='#b02a2a')
axes[0].set_title('MAE (lower better)')
sub['spearman_within_scenario'].sort_values().plot.barh(ax=axes[1], color='#1b4f9c')
axes[1].set_title('within-scenario Spearman (higher better)')
for a in axes: a.grid(alpha=.25)
plt.tight_layout(); plt.show()

## Generalisation along each shift axis

Each split changes exactly one thing relative to `test_id`, which is what makes
a gap attributable.

In [ ]:
gen = pd.read_csv(T+'generalisation.csv')
gen.round(4).to_string(index=False)

In [ ]:
from IPython.display import Image, display
import os
if os.path.exists('../results/figures/generalisation.png'):
    display(Image(filename='../results/figures/generalisation.png'))

## Is any difference real?

Paired per-row Wilcoxon with Holm-Bonferroni, against the reference-style
tabular model. **This conditions on one trained model per method** — it answers
a question about two sets of weights, not about two methods. The method-level
question needs the seed study in notebook 3.

In [ ]:
st = pd.read_csv(T+'statistical_tests.csv')
s = st[st.split=='test_id'][['name_a','mean_a','mean_b','difference','ci_lower','ci_upper','p_adjusted','effect_size','significant']]
s.round(5).to_string(index=False)